In [1]:
!pip install pyarrow


In [2]:
import pickle

In [3]:
import pandas as pd

In [4]:
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_squared_error

In [6]:
pd.__version__

'1.4.2'

In [7]:
def read_dataframe(filename):

    df = pd.read_parquet(filename)
    df.columns

    df.lpep_pickup_datetime = pd.to_datetime(df.tpep_pickup_datetime)
    df.lpep_dropoff_datetime = pd.to_datetime(df.tpep_dropoff_datetime)

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    numerical = ['trip_distance']

    df[categorical] = df[categorical].astype(str)

    return df


In [8]:
df_train = read_dataframe('./data/yellow_tripdata_2023-01.parquet')
df_val = read_dataframe('./data/yellow_tripdata_2023-02.parquet')


/tmp/ipykernel_37758/500405987.py:6: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df.lpep_pickup_datetime = pd.to_datetime(df.tpep_pickup_datetime)
/tmp/ipykernel_37758/500405987.py:7: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df.lpep_dropoff_datetime = pd.to_datetime(df.tpep_dropoff_datetime)
/tmp/ipykernel_37758/500405987.py:6: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df.lpep_pickup_datetime = pd.to_datetime(df.tpep_pickup_datetime)
/tmp/ipykernel_37758/500405987.py:7: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-a

In [ ]:
df_train.columns

## Number of columns :
## 19

In [ ]:
len(df_train), len(df_val)

In [ ]:
std_dv_jan = df_train['duration'].std()

In [ ]:
std_dv_jan

## Standard Deviation in january

# 42.59

In [ ]:
train_before = 3066766
train_after = 3009173

val_before = 2973955
val_after = 2855951

train_fraction = train_after / train_before
val_fraction = val_after / val_before

train_fraction, val_fraction

## Fraction after remove outlayers

# train 0.98 
# val 0.96

In [ ]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [20]:

categorical = ['PULocationID', 'DOLocationID']

#categorical = ['PU_DO']   #  ['PULocationID', 'DOLocationID']
#numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical].to_dict(orient='records')

#train_dicts = df_train[categorical + numerical].to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)

#val_dicts = df_val[categorical + numerical].to_dict(orient='records')

val_dicts = df_val[categorical].to_dict(orient='records')



X_val = dv.transform(val_dicts)



In [21]:
len(dv.get_feature_names_out())

515

## Demensionality of this matrix

# 515

In [22]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values
y_val_self = df_train[target].values

In [23]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)
y_pred_self = lr.predict(X_val_self)

In [25]:
mean_squared_error(y_val_self, y_pred_self, squared=False)

7.6492610279057605

## RMSE using the same dataframe to validade 
# 7.64

In [13]:
mean_squared_error(y_val, y_pred, squared=False)

7.81183265470218

## Model Evaluation
# 7.81


In [16]:
with open('models/lin_reg.bin', 'wb' ) as f_out:
    pickle.dump((dv, lr), f_out)
    

In [ ]:
lr = Lasso(alpha=0.001)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

mean_squared_error(y_val, y_pred, squared=False)

In [ ]:
lr = Ridge()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

mean_squared_error(y_val, y_pred, squared=False)

In [ ]:
train_dicts = df[categorical + numerical].to_dict(orient='records')

In [ ]:
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

In [ ]:
X_train

In [ ]:
dv.feature_names_

In [ ]:
target = 'duration'
y_train = df[target].values

In [ ]:
y_train

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

In [ ]:
y_pred = lr.predict(X_train)

In [ ]:
sns.distplot(y_pred, label='prediction')
sns.distplot(y_train, label='actual')

plt.legend()

In [ ]:
mean_squared_error(y_train, y_pred, squared=False)

In [ ]:
ls = Lass()
lr=s.fit(X_train, y_train)

In [ ]:
y_pred = ls.predict(X_train)